In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
import json
from pathlib import Path
from typing import Literal

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from tqdm import tqdm
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch.nn.functional as F

from src.global_settings import settings
from src.features import load_score_data
from src.utils.metrics import calculate_pcc
from src.auto_explainer.explanations import FeatureExplanations
from src.utils.metrics import calculate_generality, calculate_discrimination, calculate_determination

In [ ]:
META_DIR = Path(settings.feature_meta_dir)
RECORD_DIR = Path(settings.feature_record_dir)
EXPLANATION_DIR = Path(settings.feature_explanation_dir)

In [ ]:
MODEL = "f3072_k8"
CHECKPOINT = f"{MODEL}_alpha"

LAYER = 21

# meta_df = pd.read_csv(settings.text_dataset_path)
meta_df = pd.read_csv(str(settings.text_dataset_path))

scores_data = load_score_data(checkpoint=CHECKPOINT, layer=LAYER, metric_name="total_post")
scores_df = pd.DataFrame(scores_data)

sample_counts = (scores_df > 0).sum(axis=0)
sample_counts = sample_counts[sample_counts > 0]

MIN_ACTIVE_SAMPLES = 200
live_feature_indices = sample_counts[sample_counts > MIN_ACTIVE_SAMPLES].index.tolist()

scores_df = scores_df[live_feature_indices]

print(f"[{CHECKPOINT}_L{LAYER}] Live Feeatures: {len(live_feature_indices)}")

scores_df

### Types of Features

|     type        | pcc_label | pcc_pred |
| --------------- | :-------: | :------: |
| Human-side      | < 0       | < 0      |
| Human-like      | > 0       | < 0      |
| AI-side         | > 0       | > 0      |
| AI-like         | < 0       | > 0      |

In [ ]:
tn_features = []
fn_features = []
tp_features = []
fp_features = []

for feature_index in live_feature_indices:
    pcc_label = calculate_pcc(scores_df=scores_df, meta_df=meta_df, feature_index=feature_index, key="label")
    pcc_pred = calculate_pcc(scores_df=scores_df, meta_df=meta_df, feature_index=feature_index, key="pred")
    
    feat_pcc = {
        "feature_index": feature_index,
        "pcc_label": pcc_label,
        "pcc_pred": pcc_pred,
    }

    if pcc_label < 0 and pcc_pred < 0:
        tn_features.append(feat_pcc)
    elif pcc_label >= 0 and pcc_pred < 0:
        fn_features.append(feat_pcc)
    elif pcc_label >= 0 and pcc_pred >= 0:
        tp_features.append(feat_pcc)
    elif pcc_label < 0 and pcc_pred >= 0:
        fp_features.append(feat_pcc)

In [ ]:
print(f" TP: {len(tp_features)} \n TN: {len(tn_features)} \n FP: {len(fp_features)} \n FN: {len(fn_features)}")

In [ ]:
fn_features

In [ ]:
def read_explanation(checkpoint: str, layer: int, feature_index: int) -> tuple[str, str]:
    explanation_path = EXPLANATION_DIR / checkpoint / str(layer) / f"{feature_index}.jsonl"
    with open(explanation_path, "r") as f:
        lines = f.readlines()
        if len(lines) < 2:
            return "", ""
        explanation = lines[0].strip()
        description = lines[1].strip()
    return explanation, description

In [ ]:
for feature in fn_features:
    exp, des = read_explanation(CHECKPOINT, LAYER, feature["feature_index"])
    print(f"Feature #{feature["feature_index"]}")
    print(f" Explanation: {exp}")
    print(f" Description: {des}")

In [ ]:
all_features = tn_features + fn_features + tp_features + fp_features
df = pd.DataFrame(all_features)

df["pcc_bias"] = df["pcc_label"] - df["pcc_pred"]

df

In [ ]:
# pcc_bias bar plot
plt.figure(figsize=(10, 6))
plt.bar(df["feature_index"], df["pcc_bias"], alpha=0.7)
plt.xlabel("Feature Index")
plt.ylabel("PCC Bias (pcc_label - pcc_pred)")
plt.title(f"PCC Bias for Features at {CHECKPOINT} Layer {LAYER}")
plt.axhline(0, color='red', linestyle='--')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# pcc bias violin plot
plt.figure(figsize=(10, 6))
plt.violinplot(df["pcc_bias"], showmeans=True)
plt.ylabel("PCC Bias (pcc_label - pcc_pred)")
plt.title(f"PCC Bias Distribution for Features at {CHECKPOINT} Layer {LAYER}")
plt.xticks([1], ['Features'])
plt.tight_layout()
plt.show()

In [ ]:
df = df.sort_values("feature_index").reset_index(drop=True)

In [ ]:
def get_color(val):
    return '#1f77b4' if val < 0 else '#d62728'

label_colors = df["pcc_label"].apply(get_color)
pred_colors = df["pcc_pred"].apply(get_color)

x = np.arange(len(df))

plt.figure(figsize=(16, 6))
ax = plt.gca()
width = 0.4
ax.bar(x - width/2, df["pcc_label"], width=width, color=label_colors, alpha=0.8, label="label")
ax.bar(x + width/2, df["pcc_pred"], width=width, color=pred_colors, alpha=0.5, label="pred")
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_title("Type of Features (Human or AI)", fontsize=14)
ax.set_xlabel("Feature Index", fontsize=12)
ax.set_ylabel("Pearson Correlation (r)", fontsize=12)
ax.legend()
ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=30))
# plt.xlim(-1, len(df))
plt.xticks(rotation=45)
# plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
scores_df

## 점수 분포

In [ ]:
feature_index = live_feature_indices[-1]

sns.histplot(scores_df[feature_index], bins=50, kde=True)
plt.title(f"Distribution of feature: {feature_index}")
plt.xlabel("Score")
plt.ylabel("Count")
plt.show()

In [ ]:
feats = scores_df.columns[:10]

melted = scores_df[feats].melt(var_name="feature_index", value_name="score")
sns.FacetGrid(melted, col="feature_index", col_wrap=5, sharex=False, sharey=False).map(sns.histplot, "score", bins=30)
plt.show()

In [ ]:
sns.violinplot(data=scores_df[feats], orient='h', density_norm='width')
plt.title("Feature Score Distributions")
plt.xlabel("Score")
plt.show()

In [ ]:
from dataclasses import dataclass
from scipy.stats import norm


def plot_score_dist(
    scores_df: pd.DataFrame,
    meta_df: pd.DataFrame,
    feature_index: int,
    label: Literal["label", "pred"] = "label",
    color_map: dict | None = None,
    bins: str | int | None = "auto",
    density: bool = True,
    show_pdf: bool = True,
):
    if color_map is None:
        color_map = {
            0: (144/255, 238/255, 144/255, 0.7),  # light green
            1: (240/255, 128/255, 128/255, 0.7),  # light coral
        }
    df = scores_df[[feature_index]].join(meta_df[[label]], how="inner")

    plt.figure()
    for val in sorted(df[label].unique()):
        subset = df[df[label] == val][feature_index]
        bins = np.histogram_bin_edges(subset, bins=bins)
        plt.hist(subset, bins=bins, alpha=0.6, label=f"{label}={val}", color=color_map[val], density=density)
        
        if show_pdf:
            mu, sigma = subset.mean(), subset.std()
            xs = np.linspace(subset.min(), subset.max(), 200)
            plt.plot(xs, norm.pdf(xs, mu, sigma), color=color_map.get(val, "gray"), lw=2)

    plt.title(f"Feature #{feature_index} score distribution by {label}")
    plt.xlabel("Feature Score")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()


@dataclass
class CIInterval:
    lower: float
    upper: float


def compute_bayesian_prob(
    feature_score: float,
    scores_df: pd.DataFrame,
    meta_df: pd.DataFrame,
    feature_index: int,
    label: Literal["label", "pred"] = "label",
    prior: float | None = None,
    ci: bool = True,
    ci_level: float = 0.95,
    num_samples: int = 10000,
) -> float | tuple[float, CIInterval]:
    df = scores_df[[feature_index]].join(meta_df[[label]], how="inner")
    groups = {val: df[df[label] == val][feature_index] for val in [0, 1] if val in df[label].unique()}
    stats = {val: (grp.mean(), grp.std() + 1e-8, len(grp)) for val, grp in groups.items()}

    if prior is None:
        counts = df[label].value_counts(normalize=True)
        prior_1 = counts.get(1, 0.5)
    else:
        prior_1 = prior
    prior_0 = 1 - prior_1
    
    p_score_1 = norm.pdf(feature_score, stats[1][0], stats[1][1]) if 1 in stats else 1e-8
    p_score_0 = norm.pdf(feature_score, stats[0][0], stats[0][1]) if 0 in stats else 1e-8
    numerator = p_score_1 * prior_1
    denominator = numerator + p_score_0 * prior_0
    posterior_1 = numerator / denominator if denominator > 0 else np.nan

    # CI estimation (using Monte Carlo simulation)
    if ci:
        mu0, sigma0, n0 = stats.get(0, (0, 1, 1))
        mu1, sigma1, n1 = stats.get(1, (0, 1, 1))

        sampled_mu0 = np.random.normal(mu0, sigma0 / np.sqrt(n0), size=num_samples)
        sampled_mu1 = np.random.normal(mu1, sigma1 / np.sqrt(n1), size=num_samples)
        sampled_sigma0 = np.abs(np.random.normal(sigma0, sigma0 / np.sqrt(2 * n0), size=num_samples))
        sampled_sigma1 = np.abs(np.random.normal(sigma1, sigma1 / np.sqrt(2 * n1), size=num_samples))


        post_samples = []
        for m0, s0, m1, s1 in zip(sampled_mu0, sampled_sigma0, sampled_mu1, sampled_sigma1):
            p1 = norm.pdf(feature_score, m1, s1 + 1e-8) * prior_1
            p0 = norm.pdf(feature_score, m0, s0 + 1e-8) * prior_0
            denom = p1 + p0
            post_samples.append(p1 / denom if denom > 0 else np.nan)
        post_samples = np.array(post_samples)

        ci_lower, ci_upper = np.nanquantile(post_samples, [(1 - ci_level) / 2, 1 - (1 - ci_level) / 2])
        return posterior_1, (ci_lower, ci_upper)

    return posterior_1

In [ ]:
idx = 0

In [ ]:
feature_index = live_feature_indices[idx]

feature_score = scores_df[feature_index].min()

plot_score_dist(scores_df, meta_df, feature_index=feature_index, label="label", bins="auto")

prob, ci95_interal = compute_bayesian_prob(feature_score, scores_df, meta_df, feature_index, label="label", ci=True)

print(f"P(label=1 | score={feature_score:.2f}) = {prob*100:.2f}% (95% CI: {ci95_interal[0]*100:.2f}%-{ci95_interal[1]*100:.2f}%)")

idx += 1

In [ ]:
row = meta_df.iloc[4]

print(f"Label: {row["label"]} | Pred: {row["pred"]}")

row['text']

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from src.utils.token_activation_scorer import TokenActivationScorer
from src.utils.model_loader import load_sae
from src.global_settings import settings

# ===== activation과 tokens 생성 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(settings.lm_weights_path)
lm_model = AutoModelForSequenceClassification.from_pretrained(settings.lm_weights_path).to(device)
lm_model.eval()

text = '''분석할 텍스트를 입력하세요.'''
encoded = tokenizer(text, max_length=settings.prev_seq_len, truncation=True, padding="max_length", return_tensors="pt")
input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())

with torch.no_grad():
    outputs = lm_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    activation = outputs.hidden_states[22]

# ===== 여기서부터 원래 코드 =====
# SAE 모델 로드
model = load_sae(checkpoint="f3072_k8_alpha", layer_idx=21)
scorer = TokenActivationScorer(model=model)

# 특정 feature(예: 1234번)의 토큰별 점수 계산
# activation: LM의 특정 레이어 출력 (shape: [seq_len, hidden_dim])
token_scores, summary = scorer.compute_token_scores_from_activation(
    activation=activation,
    feature_idx=84,
    tokens=tokens,
    exclude_special_tokens=True,
    return_pre_activation=True
)

# 상위 10개 토큰 확인
top_tokens = scorer.get_top_activating_tokens(
    activation=activation,
    feature_idx=84,
    top_k=10,
    attention_mask=attention_mask,
    tokens=tokens
)

print(f"Total score (post): {summary['total_score_post']}")
print(f"Active tokens: {summary['num_active_tokens']}")
for t in top_tokens:
    print(f"{t['token_str']}: {t['token_score']}")

In [ ]:
import pandas as pd

text_df = pd.read_csv(settings.text_dataset_path)

text = text_df.iloc[12740]['text']

print(text)

token_scores, summary = scorer.compute_token_scores(
    text=text,
    feature_idx=1879,
    layer_idx=21,
    lm_model=lm_model,
)

active_tokens = [tok for tok in token_scores if tok["token_score"] > 0]

active_tokens = sorted(active_tokens, key=lambda x: x["token_score"], reverse=True)

active_tokens

In [ ]:
def load_feature_explanation(
    checkpoint: str, layer: int, feature_index: int, postprocess: bool = True,
) -> tuple[str, str]:
    """Load explanation and description for a feature."""
    explanation_path = (
        EXPLANATION_DIR / checkpoint / str(layer) / f"{feature_index}.jsonl"
    )
    try:
        feature_explanations = FeatureExplanations.load_from_jsonl(explanation_path)
        explanation = (
            feature_explanations.get_best_explanation().explanation
            if feature_explanations.get_best_explanation()
            else "N/A"
        )
        description = (
            feature_explanations.explanations[1].explanation
            if len(feature_explanations.explanations) > 1
            else ""
        )
    except FileNotFoundError:
        explanation = "N/A"
        description = ""

    if postprocess:
        explanation = explanation.replace("**", "").strip()
        description = description.replace("**", "").strip()

    return explanation, description

In [ ]:
# model = AutoModelForSequenceClassification.from_pretrained(
#     settings.lm_weights_path
# )
# tokenizer = AutoTokenizer.from_pretrained(settings.lm_weights_path)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = model.to(device)
# model.eval()


# logits = []
# with torch.no_grad():
#     for text in tqdm(meta_df["text"], total=len(meta_df)):
#         encoded = tokenizer(
#             text,
#             return_tensors="pt",
#             truncation=True,
#             max_length=1024,
#             padding=True,
#         )
#         input_ids = encoded["input_ids"].to(device)
#         attention_mask = encoded["attention_mask"].to(device)

#         outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#         logit = outputs.logits
#         prob = F.softmax(logit, dim=-1)[0, 1].item()
#         logits.append(prob)

# probs = pd.Series(logits, index=meta_df.index)

# meta_df.to_csv(str(settings.text_dataset_path), index=False)

# meta_df["logit"] = probs
# meta_df.head()

In [ ]:
def build_feature_df(scores_df: pd.DataFrame, meta_df: pd.DataFrame, checkpoint: str, layer: int):
    feature_records = []

    for feature_index in tqdm(scores_df.columns, total=len(scores_df.columns)):
        generality = calculate_generality(scores_df, feature_index)
        discrimination = calculate_discrimination(scores_df, meta_df, feature_index)
        determination = calculate_determination(scores_df, meta_df, feature_index)
        
        explanation, description = load_feature_explanation(checkpoint, layer, feature_index)

        feature_records.append({
            "feature_index": feature_index,
            "explanation": explanation,
            "description": description,
            "generality": generality,
            "discrimination": discrimination,
            "determination": determination,
        })

    feature_df = pd.DataFrame(feature_records)
    return feature_df

In [ ]:
feature_df = build_feature_df(scores_df, meta_df, CHECKPOINT, LAYER)

In [ ]:
feature_df.head()